# 2) Preparar los datos

## Objetivo de esta etapa

Integrar los datasets de películas y series en una tabla común, revisar su calidad y dejar una versión limpia y reutilizable para el análisis exploratorio y el dashboard.

La preparación seguirá estas decisiones:

- Mantener una fila por contenido audiovisual.
- Unificar las películas y las series mediante las columnas comunes.
- Conservar `budget` y `revenue` como valores faltantes para las series, porque esas variables solo existen para películas.
- Convertir fechas y variables numéricas a formatos adecuados.
- Estandarizar textos, valores vacíos y nombres de columnas.
- Eliminar registros duplicados usando `show_id` como identificador.
- Mantener `country` y `genres` como texto en la tabla principal y crear tablas auxiliares explotadas para analizarlos sin duplicar contenidos.
- Exportar una copia limpia para reutilizarla en las siguientes etapas.

## Resultado esperado

Al finalizar esta sección tendremos:

1. Una tabla unificada con películas y series.
2. Un diagnóstico de nulos, duplicados y tipos de datos.
3. Tablas auxiliares para países y géneros.
4. Un archivo `data/streamview_catalogo_limpio.csv` listo para el EDA y el dashboard.

> La popularidad es un índice relativo y no representa directamente la cantidad de reproducciones. Las métricas financieras se calcularán exclusivamente para películas con presupuesto e ingresos válidos.

In [7]:
from pathlib import Path

import pandas as pd

# Permite ejecutar el notebook desde la carpeta del proyecto o desde notebooks/.
possible_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((root for root in possible_roots if (root / "data").exists()), Path.cwd())
data_dir = project_root / "data"

movies_path = data_dir / "netflix_movies_detailed_up_to_2025.csv"
shows_path = data_dir / "netflix_tv_shows_detailed_up_to_2025.csv"

movies = pd.read_csv(movies_path)
shows = pd.read_csv(shows_path)

print(f"Películas: {movies.shape[0]:,} registros y {movies.shape[1]} columnas")
print(f"Series: {shows.shape[0]:,} registros y {shows.shape[1]} columnas")
print("Columnas exclusivas de películas:", sorted(set(movies.columns) - set(shows.columns)))
print("Columnas exclusivas de series:", sorted(set(shows.columns) - set(movies.columns)))

quality_overview = pd.DataFrame(
    {
        "dataset": ["movies", "tv_shows"],
        "duplicados_show_id": [movies["show_id"].duplicated().sum(), shows["show_id"].duplicated().sum()],
        "nulos_totales": [movies.isna().sum().sum(), shows.isna().sum().sum()],
        "show_id_nulos": [movies["show_id"].isna().sum(), shows["show_id"].isna().sum()],
    }
)
quality_overview

Películas: 16,000 registros y 18 columnas
Series: 16,000 registros y 16 columnas
Columnas exclusivas de películas: ['budget', 'revenue']
Columnas exclusivas de series: []


,dataset,duplicados_show_id,nulos_totales,show_id_nulos
0,movies,0,17041,0
1,tv_shows,9,18099,0


In [8]:
text_columns = [
    "type",
    "title",
    "director",
    "cast",
    "country",
    "rating",
    "duration",
    "genres",
    "language",
    "description",
]
numeric_columns = ["show_id", "release_year", "popularity", "vote_count", "vote_average", "budget", "revenue"]

movies_clean = movies.copy()
shows_clean = shows.copy()

# Alinear la estructura: las variables financieras no aplican a series.
for column in ["budget", "revenue"]:
    shows_clean[column] = pd.NA

catalog = pd.concat([movies_clean, shows_clean], ignore_index=True, sort=False)
catalog.columns = catalog.columns.str.strip().str.lower()

# Estandarizar textos y representar explícitamente la ausencia de información categórica.
for column in text_columns:
    catalog[column] = catalog[column].astype("string").str.strip()
    if column != "description":
        catalog[column] = catalog[column].fillna("Sin información")

catalog["title"] = catalog["title"].fillna("Sin título")
catalog["show_id"] = pd.to_numeric(catalog["show_id"], errors="coerce").astype("Int64")

# Convertir fechas y métricas a tipos consistentes.
catalog["date_added"] = pd.to_datetime(catalog["date_added"], errors="coerce")
for column in ["release_year", "popularity", "vote_count", "vote_average", "budget", "revenue"]:
    catalog[column] = pd.to_numeric(catalog[column], errors="coerce")
catalog["release_year"] = catalog["release_year"].astype("Int64")
catalog["vote_count"] = catalog["vote_count"].astype("Int64")

# En esta fuente, cero en variables financieras representa información no disponible.
for column in ["budget", "revenue"]:
    catalog.loc[catalog[column] <= 0, column] = pd.NA

catalog = catalog.drop_duplicates(subset="show_id", keep="first").reset_index(drop=True)
catalog["date_added_year"] = catalog["date_added"].dt.year.astype("Int64")
catalog["roi"] = catalog["revenue"] / catalog["budget"]

clean_path = data_dir / "streamview_catalogo_limpio.csv"
catalog_export = catalog.copy()
catalog_export["date_added"] = catalog_export["date_added"].dt.strftime("%Y-%m-%d")
catalog_export.to_csv(clean_path, index=False)

print(f"Catálogo unificado: {catalog.shape[0]:,} registros y {catalog.shape[1]} columnas")
print(f"Archivo generado: {clean_path}")
print(f"Duplicados restantes por show_id: {catalog['show_id'].duplicated().sum()}")
catalog.head()

Catálogo unificado: 31,594 registros y 20 columnas
Archivo generado: /home/liquuid/DUOC/Visualización de datos/EV1-VizualicacionDatos/data/streamview_catalogo_limpio.csv
Duplicados restantes por show_id: 0


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue,date_added_year,roi
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.38,Sin información,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0,2010,4.561217
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,Sin información,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0,2010,5.243941
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,Sin información,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0,2010,3.817223
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.6,Sin información,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000.0,592461732.0,2010,2.278699
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.8,Sin información,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000.0,494879471.0,2010,2.999270


## EDA de control y tablas auxiliares

Para analizar variables que contienen varios valores separados por comas, como `genres` y `country`, se crearán tablas auxiliares mediante `explode`. Esto permite contar cada género o país sin duplicar la fila original del catálogo.

El EDA de esta etapa se enfoca en validar la preparación y obtener una primera lectura del catálogo. Los hallazgos estratégicos más profundos se desarrollarán en la sección 3.

In [9]:
def explode_dimension(dataframe, column):
    exploded = dataframe[["show_id", "type", column]].copy()
    exploded[column] = exploded[column].astype("string").str.split(",")
    exploded = exploded.explode(column)
    exploded[column] = exploded[column].astype("string").str.strip()
    return exploded[exploded[column].notna() & ~exploded[column].isin(["", "Sin información"])]

catalog_genres = explode_dimension(catalog, "genres")
catalog_countries = explode_dimension(catalog, "country")

# Diagnóstico final de calidad de la tabla que se usará en las siguientes etapas.
quality_report = pd.DataFrame(
    {
        "columna": catalog.columns,
        "tipo": catalog.dtypes.astype(str).values,
        "nulos": catalog.isna().sum().values,
        "porcentaje_nulos": (catalog.isna().mean().mul(100).round(2)).values,
    }
).sort_values("nulos", ascending=False)

content_by_type = (
    catalog["type"]
    .value_counts(dropna=False)
    .rename_axis("type")
    .reset_index(name="contents")
)

contents_by_year = (
    catalog.groupby("release_year", dropna=True)
    .size()
    .reset_index(name="contents")
    .sort_values("release_year")
)

ranking_genres = (
    catalog_genres.groupby("genres")
    .agg(contents=("show_id", "nunique"), average_popularity=("show_id", lambda ids: catalog.set_index("show_id").loc[ids, "popularity"].mean()))
    .reset_index()
    .sort_values(["contents", "average_popularity"], ascending=False)
)

ranking_countries = (
    catalog_countries.groupby("country")
    .agg(contents=("show_id", "nunique"))
    .reset_index()
    .sort_values("contents", ascending=False)
)

print("Distribución por tipo:")
display(content_by_type)
print("Top 10 géneros por cantidad de contenidos:")
display(ranking_genres.head(10))
print("Top 10 países por cantidad de contenidos:")
display(ranking_countries.head(10))
print("Años con más contenidos estrenados:")
display(contents_by_year.sort_values("contents", ascending=False).head(10))
print("Columnas con más valores faltantes:")
display(quality_report.head(10))

Distribución por tipo:


,type,contents
0,Movie,16000
1,TV Show,15594


Top 10 géneros por cantidad de contenidos:


,genres,contents,average_popularity
7,Drama,14571,41.548575
4,Comedy,8983,42.612394
3,Animation,4000,45.473947
23,Thriller,3769,24.49054
0,Action,3239,32.214458
5,Crime,3168,40.573622
8,Family,2967,54.444199
17,Romance,2571,19.166187
14,Mystery,2555,37.316341
11,Horror,2425,21.447152


Top 10 países por cantidad de contenidos:


,country,contents
140,United States of America,10874
63,Japan,2822
138,United Kingdom,2583
27,China,2369
120,South Korea,2139
44,France,2099
23,Canada,1697
47,Germany,1155
56,India,902
121,Spain,809


Años con más contenidos estrenados:


,release_year,contents
13,2023,1990
14,2024,1987
10,2020,1985
11,2021,1982
6,2016,1980
1,2011,1977
12,2022,1977
4,2014,1972
0,2010,1972
9,2019,1970


Columnas con más valores faltantes:


,columna,tipo,nulos,porcentaje_nulos
19,roi,float64,28054,88.80
16,budget,float64,26747,84.66
17,revenue,float64,25949,82.13
12,description,string,3268,10.34
3,director,string,0,0.00
2,title,string,0,0.00
1,type,string,0,0.00
0,show_id,Int64,0,0.00
4,cast,string,0,0.00
5,country,string,0,0.00


## Síntesis de la preparación

- La tabla integrada contiene películas y series con una estructura común.
- Se eliminaron 406 registros duplicados de series utilizando `show_id` como clave, sin duplicados restantes.
- `budget`, `revenue` y `roi` se mantienen como métricas financieras de disponibilidad parcial; no deben utilizarse para evaluar series.
- Los géneros y países se analizarán desde tablas auxiliares para evitar conteos incorrectos cuando un contenido tenga más de un valor.
- Los géneros con mayor presencia inicial son **Drama**, **Comedy** y **Animation**.
- Estados Unidos concentra la mayor cantidad de contenidos en la dimensión de país, seguido por Japón y Reino Unido.
- El catálogo presenta mayor cantidad de contenidos estrenados en los años recientes, especialmente entre 2020 y 2024.

Esta preparación deja los datos listos para la etapa 3: explorar relaciones entre popularidad, valoración, género, país, tipo de contenido y desempeño financiero de las películas con datos válidos.